In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install wandb -q

import numpy as np
import random
import time
import os
import json

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import wandb
from wandb.integration.keras import WandbMetricsLogger

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from google.colab import userdata

print(f"TF version: {tf.__version__}")

wandb_api_key = userdata.get('WANDB_API_KEY_1')
wandb.login(key=wandb_api_key)


Mounted at /content/drive
TF version: 2.20.0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
SEED = 42

def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds()
print(f"✅ Seed დაფიქსირდა: {SEED}")

✅ Seed დაფიქსირდა: 42


In [3]:

data = np.load('/content/drive/MyDrive/Ketastasia/data/dataset_seq15_ready.npz')

X_train = data['X_train']
y_train_text = data['y_train']
X_val   = data['X_val']
y_val_text   = data['y_val']


le = LabelEncoder()
y_train_idx = le.fit_transform(y_train_text)
y_val_idx   = le.transform(y_val_text)


classes = list(le.classes_)
n_classes = len(classes)
n_timesteps = X_train.shape[1]   # 15 კადრი
n_features  = X_train.shape[2]   # 29 ფიჩერი

y_train = keras.utils.to_categorical(y_train_idx, num_classes=n_classes)
y_val   = keras.utils.to_categorical(y_val_idx, num_classes=n_classes)



In [4]:
# --- 3. CUSTOM ATTENTION LAYER ---
class AttentionPooling(layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionPooling, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape: (batch_size, timesteps, hidden_dim * 2)
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)
        super(AttentionPooling, self).build(input_shape)

    def call(self, x):
        # x-ის ზომა: (batch_size, timesteps, features)
        e = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1) # ალბათობები თითოეული კადრისთვის
        output = x * a
        return tf.reduce_sum(output, axis=1) # შეწონილი ჯამი დროის ღერძზე

    def get_config(self):
        return super(AttentionPooling, self).get_config()

In [5]:
class MacroF1EarlyStopping(keras.callbacks.Callback):
    def __init__(self, X_val, y_val_idx, patience=10, restore_best_weights=True):
        super().__init__()
        self.X_val = X_val
        self.y_val_idx = y_val_idx
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_f1 = -1
        self.wait = 0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
        f1 = classification_report(self.y_val_idx, y_pred, output_dict=True, zero_division=0)['macro avg']['f1-score']

        # ვამატებთ logs-ში, რომ Wandb-მა ავტომატურად დალოგოს
        logs['val_f1_macro'] = f1
        print(f" — val_f1_macro: {f1:.4f}")

        if f1 > self.best_f1:
            self.best_f1 = f1
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                print(f"\n🛑 ადრეული გაჩერება! საუკეთესო Val F1-Macro: {self.best_f1:.4f}")
                if self.restore_best_weights and self.best_weights:
                    self.model.set_weights(self.best_weights)

In [8]:
def build_bilstm_attention(hidden=64, dropout_rate=0.2, l2_reg=0.0005, lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),

        # BiLSTM Layer კორექტული L2 რეგულარიზაციით
        layers.Bidirectional(layers.LSTM(
            hidden,
            return_sequences=True,
            dropout=dropout_rate,
            kernel_regularizer=keras.regularizers.l2(l2_reg)
        )),

        layers.BatchNormalization(),

        # დინამიური Attention მექანიზმი
        AttentionPooling(),

        # Dense კლასიფიკატორი
        layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        layers.Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [10]:
# 1. გამოვთვალოთ კლასების წონები დისბალანსის დასაბალანსებლად
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_idx),
    y=y_train_idx
)
class_weight_dict = dict(enumerate(class_weights))
print("✅ კლასების წონები წარმატებით გამოითვალა.")

# 2. განახლებული მოდელის არქიტექტურა
def build_bilstm_attention(hidden=64, dropout_rate=0.4, l2_reg=0.002, lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),

        # BiLSTM — გაზრდილი რეგულარიზაციით
        layers.Bidirectional(layers.LSTM(
            hidden,
            return_sequences=True,
            dropout=dropout_rate,
            kernel_regularizer=keras.regularizers.l2(l2_reg)
        )),

        # ჯერ Attention, მერე Batch Normalization
        AttentionPooling(),
        layers.BatchNormalization(),

        # Dense კლასიფიკატორი — უფრო მაღალი Dropout-ით
        layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        layers.Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

✅ კლასების წონები წარმატებით გამოითვალა.


In [11]:
# 3. განახლებული საექსპერიმენტო ფუნქცია (დაემატა class_weight)
def run_experiment(cfg, run_name):
    print(f"\n{'='*60}\n🚀 იწყება ექსპერიმენტი: {run_name}\n{'='*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p1_bilstm_attention',
        name=run_name,
        config={**cfg, 'architecture': 'BiLSTM+Attention', 'pipeline': 'pipeline1A_13kp'},
        reinit=True
    )

    model = build_bilstm_attention(
        hidden=cfg['hidden'],
        dropout_rate=cfg['dropout'],
        l2_reg=cfg['l2'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        MacroF1EarlyStopping(X_val, y_val_idx, patience=cfg['patience']),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    t0 = time.time()
    # ყურადღება: აქ გადაეცემა class_weight_dict
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )
    train_time = time.time() - t0

    # --- შეფასება ---
    y_pred_val_probs = model.predict(X_val, verbose=0)
    y_pred_val = np.argmax(y_pred_val_probs, axis=1)

    n_samples = min(200, len(X_val))
    t_inf = time.time()
    _ = model.predict(X_val[:n_samples], verbose=0)
    inf_speed = ((time.time() - t_inf) / n_samples) * 1000

    rep_train = classification_report(y_train_idx, np.argmax(model.predict(X_train, verbose=0), axis=1), output_dict=True, zero_division=0)
    rep_val = classification_report(y_val_idx, y_pred_val, output_dict=True, zero_division=0)

    val_f1_macro = rep_val['macro avg']['f1-score']
    overfit_gap = rep_train['macro avg']['f1-score'] - val_f1_macro

    cm = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.tight_layout()
    cm_path = f'/content/{run_name}_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'training_time_sec': train_time,
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)
    model.save(f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5')

    wandb.finish()

    return {
        'run_name': run_name,
        'val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'n_params': model.count_params()
    }

In [12]:
configs_round3 = [
    # მოდელი 1: ძლიერი Dropout (0.4) და გაზრდილი L2 (0.002) ოვერფიტინგის დასამორჩილებლად
    {'hidden': 64, 'dropout': 0.4, 'l2': 0.002, 'lr': 0.001, 'batch_size': 32, 'epochs': 60, 'patience': 15},
    # მოდელი 2: ოდნავ უფრო კომპაქტური ჰიდენ ზომა, კიდევ უფრო მაღალი დროპაუტით
    {'hidden': 48, 'dropout': 0.45, 'l2': 0.0015, 'lr': 0.001, 'batch_size': 32, 'epochs': 60, 'patience': 15}
]

results = []
for i, cfg in enumerate(configs_round3):
    res = run_experiment(cfg, run_name=f"bilstm_att_h{cfg['hidden']}_round3_run_{i+1}")
    results.append(res)

# შედარების ცხრილი
df_res = pd.DataFrame(results)
print("\n📊 რაუნდი 3-ის შედეგები:")
print(df_res.to_string(index=False))


🚀 იწყება ექსპერიმენტი: bilstm_att_h64_round3_run_1


Epoch 1/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0845 - loss: 3.8687 — val_f1_macro: 0.0770
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - accuracy: 0.1024 - loss: 3.5997 - val_accuracy: 0.1379 - val_loss: 3.4547 - val_f1_macro: 0.0770 - learning_rate: 0.0010
Epoch 2/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1474 - loss: 3.3897 — val_f1_macro: 0.1055
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.1595 - loss: 3.2967 - val_accuracy: 0.1724 - val_loss: 3.3154 - val_f1_macro: 0.1055 - learning_rate: 0.0010
Epoch 3/60
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1747 - loss: 3.2378 — val_f1_macro: 0.1253
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.1779 - loss: 3.1580 - val_accuracy: 0.2293 - val_loss: 3.1571 - val_f1_macro: 0.1253 - learning_rate: 0.0010
Epoch 4/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2010 - loss: 3.1060 — val_f1_macro: 0.1339
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.1878 - loss: 3.0929 - va

epoch/accuracy,▁▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▆▆▆▇▇▇▇▇▇▇█████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,█████████████████████▄▄▄▄▄▄▄▂▂▂▂▁
epoch/loss,█▆▅▅▅▄▄▄▄▃▄▃▃▃▂▂▃▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▇▇▆▇▆██▆▅▂▆▆▆▆▅▆▄▆▅▆▅▆▅▅▆▆▄▅▅▃▃
epoch/val_loss,█▇▅▄▃▃▃▃▂▂▂▃▂▁▁▂▁▂▂▃▂▁▁▁▁▁▁▁▂▂▁▂▂
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.31565



🚀 იწყება ექსპერიმენტი: bilstm_att_h48_round3_run_2


Epoch 1/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0707 - loss: 3.7745 — val_f1_macro: 0.0736
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 51ms/step - accuracy: 0.0912 - loss: 3.5498 - val_accuracy: 0.1069 - val_loss: 3.3530 - val_f1_macro: 0.0736 - learning_rate: 0.0010
Epoch 2/60
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1207 - loss: 3.3419 — val_f1_macro: 0.1088
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.1354 - loss: 3.2450 - val_accuracy: 0.1862 - val_loss: 3.2383 - val_f1_macro: 0.1088 - learning_rate: 0.0010
Epoch 3/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1408 - loss: 3.2478 — val_f1_macro: 0.1301
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.1442 - loss: 3.1783 - val_accuracy: 0.2431 - val_loss: 3.0827 - val_f1_macro: 0.1301 - learning_rate: 0.0010
Epoch 4/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1583 - loss: 3.1686 — val_f1_macro: 0.1397
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.1633 - loss: 3.0763 - val

epoch/accuracy,▁▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆█▇▇██████
epoch/epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/learning_rate,███████████████████▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▂▁▁▂▁
epoch/val_accuracy,▁▅▇▇▅▇▇▇▇▆▇██▆█▇▆▆▅▅▅▅▄▅▅▄▄
epoch/val_loss,█▇▅▄▃▃▃▂▃▃▃▂▂▂▁▁▂▁▂▁▁▂▁▁▁▁▁
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.26667



📊 რაუნდი 3-ის შედეგები:
                   run_name  val_f1_macro  overfit_gap_f1  inference_ms_per_seq  n_params
bilstm_att_h64_round3_run_1      0.165419        0.072619              1.184310     58664
bilstm_att_h48_round3_run_2      0.191397        0.040419              0.525296     38280


In [9]:
def run_experiment(cfg, run_name):
    print(f"\n{'='*60}\n🚀 იწყება ექსპერიმენტი: {run_name}\n{'='*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p1_bilstm_attention',
        name=run_name,
        config={**cfg, 'architecture': 'BiLSTM+Attention', 'pipeline': 'pipeline1A_13kp'},
        reinit=True
    )

    model = build_bilstm_attention(
        hidden=cfg['hidden'],
        dropout_rate=cfg['dropout'],
        l2_reg=cfg['l2'],
        lr=cfg['lr']
    )

    # ვიყენებთ ჩვენს ახალ Custom EarlyStopping-ს
    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        MacroF1EarlyStopping(X_val, y_val_idx, patience=cfg['patience']),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        verbose=1
    )
    train_time = time.time() - t0

    # --- შეფასება (Inference Speed-ის და ჩემპიონი მეტრიკების გამოთვლა) ---
    y_pred_val_probs = model.predict(X_val, verbose=0)
    y_pred_val = np.argmax(y_pred_val_probs, axis=1)

    # სისწრაფის გაზომვა (Inference Speed)
    n_samples = min(200, len(X_val))
    t_inf = time.time()
    _ = model.predict(X_val[:n_samples], verbose=0)
    inf_speed = ((time.time() - t_inf) / n_samples) * 1000

    # რეპორტები
    rep_train = classification_report(y_train_idx, np.argmax(model.predict(X_train, verbose=0), axis=1), output_dict=True, zero_division=0)
    rep_val = classification_report(y_val_idx, y_pred_val, output_dict=True, zero_division=0)

    val_f1_macro = rep_val['macro avg']['f1-score']
    overfit_gap = rep_train['macro avg']['f1-score'] - val_f1_macro

    # ლოგირება მატრიცის და დამატებითი მეტრიკების
    cm = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.tight_layout()
    cm_path = f'/content/{run_name}_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'training_time_sec': train_time,
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    # მოდელის შენახვა
    os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)
    model.save(f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5')

    wandb.finish()

    return {
        'run_name': run_name,
        'val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'n_params': model.count_params()
    }

# --- 7. დაგეგმილი რაუნდის გაშვება ---
configs_round2 = [
    {'hidden': 64, 'dropout': 0.2, 'l2': 0.0005, 'lr': 0.001, 'batch_size': 32, 'epochs': 60, 'patience': 12},
    {'hidden': 32, 'dropout': 0.15, 'l2': 0.0003, 'lr': 0.001, 'batch_size': 32, 'epochs': 60, 'patience': 12}
]

results = []
for i, cfg in enumerate(configs_round2):
    res = run_experiment(cfg, run_name=f"bilstm_att_h{cfg['hidden']}_run_{i+1}")
    results.append(res)

# შედარების ცხრილი
df_res = pd.DataFrame(results)
print("\n📊 რაუნდი 2-ის შედეგები:")
print(df_res.to_string(index=False))


🚀 იწყება ექსპერიმენტი: bilstm_att_h64_run_1


Epoch 1/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1706 - loss: 3.1183 — val_f1_macro: 0.1055
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 51ms/step - accuracy: 0.2299 - loss: 2.8386 - val_accuracy: 0.2517 - val_loss: 2.9718 - val_f1_macro: 0.1055 - learning_rate: 0.0010
Epoch 2/60
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2883 - loss: 2.5284 — val_f1_macro: 0.1031
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.3037 - loss: 2.4621 - val_accuracy: 0.2586 - val_loss: 2.7557 - val_f1_macro: 0.1031 - learning_rate: 0.0010
Epoch 3/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3055 - loss: 2.4260 — val_f1_macro: 0.1417
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.3272 - loss: 2.3518 - val_accuracy: 0.2931 - val_loss: 2.5484 - val_f1_macro: 0.1417 - learning_rate: 0.0010
Epoch 4/60
89/92 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3323 - loss: 2.3242 — val_f1_macro: 0.1607
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.3412 - loss: 2.2632 - val

epoch/accuracy,▁▃▃▃▄▄▅▅▅▅▆▆▆▆▆▆▇▆▇▇▇▇▇█▇▇█▇████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████████████▄▄▄▄▂▂▂▂▂▁▁▁▁
epoch/loss,█▆▅▅▄▄▄▄▄▃▃▃▃▃▂▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▃▄▄▆▇▅▇▅▆▇▇▄▇▅▇▅▅█▆▆▇▇▆▇▇▇▆▆▆▆
epoch/val_loss,█▆▄▃▂▂▂▁▁▂▁▁▁▂▁▂▁▂▂▁▂▂▁▁▂▁▂▂▂▂▂▂
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.5466



🚀 იწყება ექსპერიმენტი: bilstm_att_h32_run_2


Epoch 1/60
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1542 - loss: 3.0662 — val_f1_macro: 0.0818
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.2327 - loss: 2.7897 - val_accuracy: 0.2207 - val_loss: 2.9594 - val_f1_macro: 0.0818 - learning_rate: 0.0010
Epoch 2/60
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3005 - loss: 2.4344 — val_f1_macro: 0.0990
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3122 - loss: 2.3874 - val_accuracy: 0.2448 - val_loss: 2.7032 - val_f1_macro: 0.0990 - learning_rate: 0.0010
Epoch 3/60
88/92 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3265 - loss: 2.3121 — val_f1_macro: 0.1228
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3347 - loss: 2.2621 - val_accuracy: 0.2621 - val_loss: 2.5527 - val_f1_macro: 0.1228 - learning_rate: 0.0010
Epoch 4/60
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3433 - loss: 2.1903 — val_f1_macro: 0.1396
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3469 - loss: 2.1691 - val

epoch/accuracy,▁▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████
epoch/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
epoch/learning_rate,█████████████████████████████████▃▃▃▃▁▁▁
epoch/loss,█▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▃▄▄▄▅▄▆▆▅▆▇▆▇▅▇▆▆▇▆▇▆▆▇█▇█▆█▇▆▆▇▆▇▆▆▆
epoch/val_loss,█▅▄▃▃▃▃▂▂▂▂▂▁▂▁▂▂▁▂▁▁▁▁▂▂▁▂▂▁▂▂▂▂▂▂▂▁▂▂▂
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.59082



📊 რაუნდი 2-ის შედეგები:
            run_name  val_f1_macro  overfit_gap_f1  inference_ms_per_seq  n_params
bilstm_att_h64_run_1      0.275702        0.154384              0.546510     58664
bilstm_att_h32_run_2      0.274829        0.235310              0.889064     21992


ოვერფიტი მოვაგვარეთ მარა გვაქ უნდერფიტი


In [13]:
import math

# 1. "დარბილებული" კლასის წონების გამოთვლა (რომ გრადიენტები არ აირიოს)
def get_smoothed_class_weights(y_idx, smooth_factor=0.15):
    counter = pd.Series(y_idx).value_counts().to_dict()
    max_val = max(counter.values())
    class_weights = {}
    for cls, count in counter.items():
        # ლოგარითმული დარბილება
        class_weights[cls] = math.log(smooth_factor * max_val + count) / math.log(count)
    return class_weights

class_weight_dict = get_smoothed_class_weights(y_train_idx)
print("✅ დარბილებული კლასების წონები:", class_weight_dict)

# 2. მოდელის სტრუქტურა LayerNormalization-ით და ზომიერი დროპაუტით
def build_bilstm_attention_v4(hidden=64, dropout_rate=0.2, l2_reg=0.0001, lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),

        # BiLSTM
        layers.Bidirectional(layers.LSTM(
            hidden,
            return_sequences=True,
            dropout=dropout_rate,
            kernel_regularizer=keras.regularizers.l2(l2_reg)
        )),

        # Batch Normalization-ის ნაცვლად ვიყენებთ LayerNormalization-ს
        layers.LayerNormalization(),

        # Attention მექანიზმი
        AttentionPooling(),

        # Dense კლასიფიკატორი
        layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        layers.Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 3. განახლებული საექსპერიმენტო ფუნქცია
def run_experiment_v4(cfg, run_name):
    print(f"\n{'='*60}\n🚀 იწყება ექსპერიმენტი: {run_name}\n{'='*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p1_bilstm_attention_v4',
        name=run_name,
        config={**cfg, 'architecture': 'BiLSTM+Attention_v4', 'pipeline': 'pipeline1A_13kp'},
        reinit=True
    )

    model = build_bilstm_attention_v4(
        hidden=cfg['hidden'],
        dropout_rate=cfg['dropout'],
        l2_reg=cfg['l2'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        MacroF1EarlyStopping(X_val, y_val_idx, patience=cfg['patience']),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )
    train_time = time.time() - t0

    # --- შეფასება ---
    y_pred_val_probs = model.predict(X_val, verbose=0)
    y_pred_val = np.argmax(y_pred_val_probs, axis=1)

    n_samples = min(200, len(X_val))
    t_inf = time.time()
    _ = model.predict(X_val[:n_samples], verbose=0)
    inf_speed = ((time.time() - t_inf) / n_samples) * 1000

    rep_train = classification_report(y_train_idx, np.argmax(model.predict(X_train, verbose=0), axis=1), output_dict=True, zero_division=0)
    rep_val = classification_report(y_val_idx, y_pred_val, output_dict=True, zero_division=0)

    val_f1_macro = rep_val['macro avg']['f1-score']
    overfit_gap = rep_train['macro avg']['f1-score'] - val_f1_macro

    cm = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.tight_layout()
    cm_path = f'/content/{run_name}_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'training_time_sec': train_time,
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)
    model.save(f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5')
    wandb.finish()

    return {
        'run_name': run_name,
        'val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'n_params': model.count_params()
    }

# 4. ახალი რაუნდის (Round 4) გაშვება ოპტიმალური ბალანსით
configs_round4 = [
    # მოდელი 1: სტანდარტული ოპტიმალური კონფიგურაცია უფრო მეტი ტევადობით (Dense 128)
    {'hidden': 64, 'dropout': 0.25, 'l2': 0.0001, 'lr': 0.001, 'batch_size': 32, 'epochs': 60, 'patience': 15},
    # მოდელი 2: ოდნავ უფრო დიდი ტევადობა (hidden 128) თუკი მონაცემები ძალიან კომპლექსურია
    {'hidden': 128, 'dropout': 0.3, 'l2': 0.0001, 'lr': 0.001, 'batch_size': 32, 'epochs': 60, 'patience': 15}
]

results = []
for i, cfg in enumerate(configs_round4):
    res = run_experiment_v4(cfg, run_name=f"bilstm_att_v4_h{cfg['hidden']}_run_{i+1}")
    results.append(res)

df_res = pd.DataFrame(results)
print("\n📊 რაუნდი 4-ის შედეგები:")
print(df_res.to_string(index=False))

✅ დარბილებული კლასების წონები: {3: 1.0234663799068302, 21: 1.0264654353641316, 14: 1.0394331645270036, 15: 1.059673108738463, 16: 1.0670243623196674, 22: 1.0744850397370254, 11: 1.076291280249231, 23: 1.077542019998467, 0: 1.0883143492770226, 8: 1.099191065004147, 13: 1.1002125786971388, 6: 1.1067859345066078, 12: 1.1116409689202773, 18: 1.1142288907694182, 19: 1.115566048941173, 17: 1.116933290157892, 1: 1.1290868920000425, 10: 1.1361099527399694, 7: 1.1399019044605696, 24: 1.1439011676870978, 20: 1.1573275006185775, 2: 1.167694701515073, 4: 1.1826584313820883, 5: 1.1929394810646674, 9: 3.7403852269827}

🚀 იწყება ექსპერიმენტი: bilstm_att_v4_h64_run_1


Epoch 1/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.1854 - loss: 3.1888 — val_f1_macro: 0.1185
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 57ms/step - accuracy: 0.2381 - loss: 2.9034 - val_accuracy: 0.2690 - val_loss: 2.4757 - val_f1_macro: 0.1185 - learning_rate: 0.0010
Epoch 2/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2874 - loss: 2.6384 — val_f1_macro: 0.1178
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.3017 - loss: 2.5858 - val_accuracy: 0.2655 - val_loss: 2.4326 - val_f1_macro: 0.1178 - learning_rate: 0.0010
Epoch 3/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3128 - loss: 2.5104 — val_f1_macro: 0.1419
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.3228 - loss: 2.4638 - val_accuracy: 0.2828 - val_loss: 2.3848 - val_f1_macro: 0.1419 - learning_rate: 0.0010
Epoch 4/60
90/92 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3436 - loss: 2.4002 — val_f1_macro: 0.1737
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.3510 - loss: 2.3609 - va

epoch/accuracy,▁▂▃▄▄▄▄▄▅▅▅▆▅▅▆▆▇▆▇▇▇▇█▇▇▇██████████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,██████████████████▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▂▄▆▇▆▇█▆█▆▅▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▆▇▇▇▆▆▆▆▆▆
epoch/val_loss,█▇▆▅▅▄▄▃▂▄▂▂▃▁▂▃▂▂▂▃▃▄▃▃▃▃▃▃▄▄▃▄▄▄▄▄▄▄▄▄
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.5398



🚀 იწყება ექსპერიმენტი: bilstm_att_v4_h128_run_2


Epoch 1/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1931 - loss: 3.1992 — val_f1_macro: 0.1400
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - accuracy: 0.2391 - loss: 2.9585 - val_accuracy: 0.2707 - val_loss: 2.5052 - val_f1_macro: 0.1400 - learning_rate: 0.0010
Epoch 2/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.2735 - loss: 2.7287 — val_f1_macro: 0.1491
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.2891 - loss: 2.6764 - val_accuracy: 0.2914 - val_loss: 2.4686 - val_f1_macro: 0.1491 - learning_rate: 0.0010
Epoch 3/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2986 - loss: 2.6160 — val_f1_macro: 0.1683
92/92 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.3136 - loss: 2.5484 - val_accuracy: 0.3000 - val_loss: 2.4266 - val_f1_macro: 0.1683 - learning_rate: 0.0010
Epoch 4/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.3307 - loss: 2.5374 — val_f1_macro: 0.1655
92/92 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step - accuracy: 0.3398 - loss: 2.4610 - va

epoch/accuracy,▁▃▃▄▄▅▅▅▆▆▇▇▇▇▇▇█████
epoch/epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
epoch/learning_rate,██████████▄▄▄▄▂▂▂▂▁▁▁
epoch/loss,█▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
epoch/val_accuracy,▁▄▅▄▇█▁▅▂▁▂▃▃▄▄▅▅▆▆▆▇
epoch/val_loss,█▇▅▆▆▁▅▂▄▃▃▅▂▄▄▄▄▅▃▅▄
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.46633



📊 რაუნდი 4-ის შედეგები:
                run_name  val_f1_macro  overfit_gap_f1  inference_ms_per_seq  n_params
 bilstm_att_v4_h64_run_1      0.243874        0.167562              0.555308     68264
bilstm_att_v4_h128_run_2      0.207114        0.086978              0.745622    198696


Conv1D + GlobalMaxPooling

In [14]:
def build_conv1d_model(filters=64, kernel_size=3, dropout_rate=0.3, l2_reg=0.0001, lr=0.003):
    model = keras.Sequential([
        layers.Input(shape=(n_timesteps, n_features)),

        # პირველი Conv1D ბლოკი
        layers.Conv1D(filters=filters, kernel_size=kernel_size, padding='same', kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.LayerNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),

        # მეორე Conv1D ბლოკი (უფრო მეტი ფილტრით)
        layers.Conv1D(filters=filters * 2, kernel_size=kernel_size, padding='same', kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.LayerNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),

        # დროის ღერძზე ყველაზე მნიშვნელოვანი ფიჩერების ამოკრეფა
        layers.GlobalMaxPooling1D(),

        # კლასიფიკატორი
        layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(l2_reg)),
        layers.Dropout(dropout_rate),
        layers.Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# განახლებული საექსპერიმენტო ფუნქცია Conv1D-სთვის
def run_experiment_conv1d(cfg, run_name):
    print(f"\n{'='*60}\n🚀 იწყება ექსპერიმენტი: {run_name}\n{'='*60}")

    wandb.init(
        project='ildolcefarniente',
        group='p2_conv1d_temporal',
        name=run_name,
        config={**cfg, 'architecture': 'Conv1D+GlobalMax', 'pipeline': 'pipeline1A_13kp'},
        reinit=True
    )

    model = build_conv1d_model(
        filters=cfg['filters'],
        kernel_size=cfg['kernel_size'],
        dropout_rate=cfg['dropout'],
        l2_reg=cfg['l2'],
        lr=cfg['lr']
    )

    callbacks = [
        WandbMetricsLogger(log_freq='epoch'),
        MacroF1EarlyStopping(X_val, y_val_idx, patience=cfg['patience']),
        # უფრო სწრაფი რეაგირება Plateau-ზე
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ]

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=callbacks,
        class_weight=class_weight_dict, # ვიყენებთ ისევ დარბილებულ წონებს
        verbose=1
    )
    train_time = time.time() - t0

    # --- შეფასება ---
    y_pred_val_probs = model.predict(X_val, verbose=0)
    y_pred_val = np.argmax(y_pred_val_probs, axis=1)

    n_samples = min(200, len(X_val))
    t_inf = time.time()
    _ = model.predict(X_val[:n_samples], verbose=0)
    inf_speed = ((time.time() - t_inf) / n_samples) * 1000

    rep_train = classification_report(y_train_idx, np.argmax(model.predict(X_train, verbose=0), axis=1), output_dict=True, zero_division=0)
    rep_val = classification_report(y_val_idx, y_pred_val, output_dict=True, zero_division=0)

    val_f1_macro = rep_val['macro avg']['f1-score']
    overfit_gap = rep_train['macro avg']['f1-score'] - val_f1_macro

    cm = confusion_matrix(y_val_idx, y_pred_val)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.tight_layout()
    cm_path = f'/content/{run_name}_cm.png'
    plt.savefig(cm_path, dpi=150)
    plt.close()

    wandb.log({
        'final_val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'training_time_sec': train_time,
        'val_confusion_matrix_img': wandb.Image(cm_path)
    })

    os.makedirs('/content/drive/MyDrive/Ketastasia/models', exist_ok=True)
    model.save(f'/content/drive/MyDrive/Ketastasia/models/{run_name}.h5')
    wandb.finish()

    return {
        'run_name': run_name,
        'val_f1_macro': val_f1_macro,
        'overfit_gap_f1': overfit_gap,
        'inference_ms_per_seq': inf_speed,
        'n_params': model.count_params()
    }

# 5. კონფიგურაციები Conv1D ტესტისთვის
configs_conv1d = [
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.3, 'l2': 0.0001, 'lr': 0.003, 'batch_size': 32, 'epochs': 60, 'patience': 15},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.35, 'l2': 0.0001, 'lr': 0.003, 'batch_size': 32, 'epochs': 60, 'patience': 15}
]

results_conv = []
for i, cfg in enumerate(configs_conv1d):
    res = run_experiment_conv1d(cfg, run_name=f"conv1d_temporal_f{cfg['filters']}_run_{i+1}")
    results_conv.append(res)

df_res_conv = pd.DataFrame(results_conv)
print("\n📊 Conv1D ექსპერიმენტის შედეგები:")
print(df_res_conv.to_string(index=False))


🚀 იწყება ექსპერიმენტი: conv1d_temporal_f64_run_1


Epoch 1/60
86/92 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1862 - loss: 3.3295 — val_f1_macro: 0.1133
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.2602 - loss: 2.8175 - val_accuracy: 0.2707 - val_loss: 2.5104 - val_f1_macro: 0.1133 - learning_rate: 0.0030
Epoch 2/60
87/92 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3385 - loss: 2.4046 — val_f1_macro: 0.2031
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3571 - loss: 2.3250 - val_accuracy: 0.3190 - val_loss: 2.3946 - val_f1_macro: 0.2031 - learning_rate: 0.0030
Epoch 3/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3733 - loss: 2.2712 — val_f1_macro: 0.2332
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3881 - loss: 2.1865 - val_accuracy: 0.3552 - val_loss: 2.3402 - val_f1_macro: 0.2332 - learning_rate: 0.0030
Epoch 4/60
87/92 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4193 - loss: 2.1005 — val_f1_macro: 0.2440
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4269 - loss: 2.0449 - val_acc

epoch/accuracy,▁▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█████████████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch/learning_rate,███████████▄▄▄▄▄▄▄▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▄▄▅▅▅▆▆▅▆▆▆▇▇▇▇▇▇▇█▇▇█████▇█▇██▇▇▇▇▇▇█
epoch/val_loss,█▆▆▅▅▃▄▃▃▃▃▂▂▂▂▁▂▂▁▂▁▂▁▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.68333



🚀 იწყება ექსპერიმენტი: conv1d_temporal_f128_run_2


Epoch 1/60
89/92 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.1429 - loss: 3.6651 — val_f1_macro: 0.1210
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.2129 - loss: 3.0456 - val_accuracy: 0.2828 - val_loss: 2.5563 - val_f1_macro: 0.1210 - learning_rate: 0.0030
Epoch 2/60
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2904 - loss: 2.5999 — val_f1_macro: 0.1316
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.3034 - loss: 2.5382 - val_accuracy: 0.2741 - val_loss: 2.4774 - val_f1_macro: 0.1316 - learning_rate: 0.0030
Epoch 3/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3257 - loss: 2.4370 — val_f1_macro: 0.1566
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.3367 - loss: 2.3905 - val_accuracy: 0.3000 - val_loss: 2.4288 - val_f1_macro: 0.1566 - learning_rate: 0.0030
Epoch 4/60
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3578 - loss: 2.3169 — val_f1_macro: 0.1887
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.3558 - loss: 2.2850 - val

epoch/accuracy,▁▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██▇███████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
epoch/learning_rate,█████████████████▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▇▆▆▆▅▅▅▄▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▂▃▃▅▄▄▄▄▅▅▇▅▆▇▅▆▅▆▇▇██▇█▇█████████████
epoch/val_loss,█▇▅▅▅▄▅▄▃▃▃▃▂▂▃▃▂▃▃▂▁▁▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁
final_val_f1_macro,▁
inference_ms_per_seq,▁
overfit_gap_f1,▁
training_time_sec,▁
epoch/accuracy,0.6017



📊 Conv1D ექსპერიმენტის შედეგები:
                  run_name  val_f1_macro  overfit_gap_f1  inference_ms_per_seq  n_params
 conv1d_temporal_f64_run_1      0.344755        0.405680              0.458902     50457
conv1d_temporal_f128_run_2      0.351115        0.335765              0.842236    146713
